# Qwen3.7-Plus Virginia Global — same 24 drawings

This notebook uses the integrated GitHub source; no embedded ZIP/base64 or personal workspace ID.
It creates a new run, leaving the submitted historical evidence unchanged. Configure your uploaded branch
in Cell 1. All paid calls default to OFF. A fresh runtime has no historical response cache and may cost
more than the recorded cached evaluation. Model responses may differ. Human validation remains pending.


In [ ]:
# @title Cell 1 — clone your integration branch and install; no model calls
import os
import shutil
import subprocess
import sys
from pathlib import Path

BRANCH = "zhetao/qwen37plus-same24"  # @param {type:"string"}

ROOT = Path("/content/drawing-intelligence-plus37")
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv")
if not ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            "https://github.com/ted-cc6/drawing-intelligence.git",
            str(ROOT),
        ],
        check=True,
    )
else:
    actual = subprocess.check_output(["git", "branch", "--show-current"], cwd=ROOT, text=True).strip()
    assert actual == BRANCH, f"Existing checkout is on {actual}; use the intended branch."
assert (ROOT / "spikes/qwen37plus/run.py").exists(), "Upload the integration files to this branch first."
subprocess.run([UV, "sync", "--locked"], cwd=ROOT, check=True)

In [ ]:
# @title Cell 2 — inspect historical results; no API calls
subprocess.run([UV, "run", "python", "spikes/qwen37plus/verify_results.py"], cwd=ROOT, check=True)

In [ ]:
# @title Cell 3 — new run; paid execution OFF by default
QWEN_BASE_URL = ""  # @param {type:"string"}
RUN_PAID = False  # @param {type:"boolean"}
SOFT_CAP_USD = 1.00  # @param {type:"number"}
PAYER_NAME = "Zhetao Zhang"  # @param {type:"string"}
env = {**os.environ, "QWEN_BASE_URL": QWEN_BASE_URL.strip()}
args = [UV, "run", "python", "spikes/qwen37plus/run.py", "--budget-usd", str(SOFT_CAP_USD)]
if RUN_PAID:
    key = os.environ.get("PERSONAL_QWEN_API_KEY", "").strip()
    if not key:
        try:
            from google.colab import userdata

            key = (userdata.get("PERSONAL_QWEN_API_KEY") or "").strip()
        except Exception:
            pass
    if not key:
        from getpass import getpass

        key = getpass("Personal Qwen API key (hidden): ").strip()
    assert key and PAYER_NAME.strip()
    env["QWEN_API_KEY"] = key
    args += [
        "--run",
        "--approval-note",
        (
            f"{PAYER_NAME}: personally authorize a separate Plus same-24 run; "
            f"cumulative evaluator soft cap USD {SOFT_CAP_USD:.2f}; "
            "historical and unknown charges additional."
        ),
    ]
try:
    subprocess.run(args, cwd=ROOT, env=env, check=True)
finally:
    env.pop("QWEN_API_KEY", None)
    if "key" in globals():
        del key

In [ ]:
# @title Cell 4 — download PRIVATE rerun checkpoint; no API calls
import sqlite3
import zipfile
from datetime import UTC, datetime

from google.colab import files

OUT = ROOT / "spikes/qwen37plus/out/run24"
assert (OUT / "manifest.json").exists(), "Prepare or run Cell 3 first."
target = Path("/content") / (
    "Qwen37Plus_Private_Rerun_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + ".zip"
)
backup = Path("/content/plus37_private_telemetry.db")
db = ROOT / "data/telemetry.db"
if db.exists():
    with sqlite3.connect(db) as src, sqlite3.connect(backup) as dst:
        src.backup(dst)
with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, "results/" + p.relative_to(OUT).as_posix())
    for p in (ROOT / "data/cache").glob("*.json"):
        z.write(p, "data/cache/" + p.name)
    if db.exists():
        z.write(backup, "data/telemetry.db")
    for p in (OUT.parent / (OUT.name + "_config")).rglob("*"):
        if p.is_file():
            z.write(p, "config/" + p.relative_to(OUT.parent / (OUT.name + "_config")).as_posix())

files.download(str(target))
print("Private runtime backup. This notebook does not implement cross-runtime checkpoint restoration.")